# Metro Manila shapefile → CSV (step-by-step)

This notebook reproduces the logic in `metro-manila-shp-to-csv.py` with explanatory notes for each section. It computes, for each barangay in the Manila boundary, the flooded area (in sqm) and the percent of the barangay area flooded for two return periods (5yr and 25yr), then writes a CSV output.

## 1 — Imports and environment checks
We import `os`, `pandas` and `geopandas`. If packages are missing, the original script printed an instruction and re-raised the error; we keep a clear failure here so you can install missing packages.
Run this cell after activating your environment (conda/venv) that has `geopandas` and `pandas`.

In [ ]:
import os
import pandas as pd
import geopandas as gpd

# If this import block fails, install geopandas + pandas in your environment.

## 2 — Paths, constants and files
Define file locations, output path, area CRS (EPSG), and which city to filter. Modify `CITY_FILTER` to `None` to process all barangays in the shapefile.

In [ ]:
script_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else '.'
admin_file = os.path.join(script_dir, 'phl_admin4.shp')
flood_files = {
    '5yr': os.path.join(script_dir, 'MetroManila_Flood_5year.shp'),
    '25yr': os.path.join(script_dir, 'MetroManila_Flood_25year.shp'),
}
output_csv = os.path.join(script_dir, 'MetroManila_Barangay_Flood_Area_MultiRP.csv')
CITY_FILTER = 'CITY OF MANILA'  # set to None for all barangays
AREA_EPSG = 32651  # projected CRS for area calculation (meters)

script_dir, admin_file, output_csv

## 3 — Load admin shapefile and detect barangay column
We load the `phl_admin4.shp` file. The script attempts to find a suitable barangay name column by testing a small list of common column names. If none are found, it raises a helpful error.

In [ ]:
print('Loading admin shapefile...')
admin = gpd.read_file(admin_file)
# Try common column names for barangay name
brgy_col = next((c for c in ['adm4_name', 'ADM4_EN', 'adm4_en', 'barangay'] if c in admin.columns), None)
if not brgy_col:
    raise ValueError(f'Could not find barangay column in {list(admin.columns)}')
brgy_col

## 4 — Optional city filtering and reprojection
If `CITY_FILTER` is set, we detect a city column similarly and filter to that city. Then we reproject to a projected CRS (`AREA_EPSG`) and compute barangay areas in square meters.

In [ ]:
# Apply optional city filter
if CITY_FILTER:
    city_col = next((c for c in ['adm3_name', 'ADM3_EN', 'adm3_en', 'city'] if c in admin.columns), None)
    if not city_col:
        raise ValueError(f'Could not find city column in {list(admin.columns)}')
    admin = admin.loc[
        admin[city_col].astype(str).str.upper().str.strip() == CITY_FILTER.upper(),
        [brgy_col, 'geometry']
    ].copy()
    print(f'Filtered to {len(admin)} barangays in {CITY_FILTER}')
else:
    admin = admin[[brgy_col, 'geometry']].copy()

# Reproject and compute area in sqm
admin = admin.to_crs(epsg=AREA_EPSG)
admin['brgy_area_sqm'] = admin.geometry.area
admin.head(3)

## 5 — Create Manila boundary for clipping flood layers
We dissolve the set of barangay geometries to form a single boundary used to clip large flood layers before spatial operations, reducing computation and avoiding irrelevant matches.

In [ ]:
manila_boundary = admin.dissolve()[['geometry']].copy()
manila_boundary.geometry.is_valid.all()

## 6 — Prepare base `result` table
We create a pandas DataFrame listing barangay names and their areas (sqm). We'll merge flood area results into this table.

In [ ]:
result = pd.DataFrame({
    'barangay': admin[brgy_col].values,
    'brgy_area_sqm': admin['brgy_area_sqm'].values,
})
result.head(3)

## 7 — Flood processing loop (per return period)
For each flood shapefile (5yr and 25yr), we: 
- Read the flood geometry
- Reproject to match the admin CRS
- Clip the flood layer to the Manila boundary to avoid outside polygons
- Spatially join flood polygons to candidate barangays (intersects)
- Intersect polygon pairs and compute intersection area (sqm)
- Aggregate flood area by barangay and merge into `result`
- Compute `flood_pct_{rp}` as (flood_area / brgy_area) * 100 — this is the `PCT` you asked about: percent flooded.

In [ ]:
from geopandas import GeoSeries

for rp, flood_path in flood_files.items():
    print(f'
Processing {rp}: {flood_path}')

    flood = gpd.read_file(flood_path)[['geometry']].copy()
    flood = flood.to_crs(admin.crs)
    flood = flood.loc[flood.geometry.notna() & ~flood.geometry.is_empty].copy()
    flood = gpd.clip(flood, manila_boundary)

    if flood.empty:
        result[f'flood_area_{rp}_sqm'] = 0.0
        result[f'flood_pct_{rp}'] = 0.0
        continue

    joined = gpd.sjoin(
        flood,
        admin[[brgy_col, 'geometry']],
        how='inner',
        predicate='intersects'
    )

    if joined.empty:
        flood_by_brgy = pd.DataFrame(columns=['barangay', f'flood_area_{rp}_sqm'])
    else:
        flood_geoms = joined.geometry.reset_index(drop=True)
        admin_geoms = admin.loc[joined['index_right'], 'geometry'].reset_index(drop=True)
        inter_area = flood_geoms.intersection(admin_geoms).area

        joined = joined.reset_index(drop=True)
        joined['barangay'] = admin.loc[joined['index_right'], brgy_col].reset_index(drop=True).values
        joined[f'flood_area_{rp}_sqm'] = inter_area.values

        flood_by_brgy = joined.groupby('barangay', as_index=False)[f'flood_area_{rp}_sqm'].sum()

    result = result.merge(flood_by_brgy, on='barangay', how='left')
    result[f'flood_area_{rp}_sqm'] = result[f'flood_area_{rp}_sqm'].fillna(0.0)

    # Percent flooded (PCT) — percent of barangay area covered by flood polygons
    result[f'flood_pct_{rp}'] = result[f'flood_area_{rp}_sqm'] / result['brgy_area_sqm'] * 100

print('
Flood processing complete (columns added to `result`).')

## 8 — Finalize and write CSV
We select the final columns, sort by barangay name, reset the index, and write the CSV. The resulting CSV has columns for `flood_area_*_sqm` and `flood_pct_*` (percent flooded) for each return period.

In [ ]:
final_cols = ['barangay', 'brgy_area_sqm']
for rp in flood_files:
    final_cols += [f'flood_area_{rp}_sqm', f'flood_pct_{rp}']

final = result[final_cols].sort_values('barangay').reset_index(drop=True)
final.to_csv(output_csv, index=False)
print(f'
Done. Saved: {output_csv}')
final.head(10)

## Notes and validation tips
- `flood_pct_{rp}` is the percent of each barangay (by area) that is flooded for return period `rp`. A value of 100 means the entire barangay is flooded; 0 means none.
- If `brgy_area_sqm` is zero (degenerate geometry), the division will produce `inf` or `NaN` — check `final` for odd values.
- Large flood shapefiles can be slow; clipping to `manila_boundary` reduces work.
- Verify CRS units: `AREA_EPSG` should be a projected CRS in meters for correct sqm results.